### 模型微调：unsloth

- 加载预训练模型

In [33]:
#从 unsloth 库中导入 FastLanguageModel 类。这个类提供了快速加载和推理大型语言模型的功能。
from unsloth import FastLanguageModel
import torch  # 导入torch工具，用于处理模型的数学运算

#设置最大序列长度为 2048。这意味着输入序列的最大长度将被限制为 2048 个 token。
max_seq_length = 2048 
#设置数据类型为 None。设置数据类型，让模型自动选择最适合的精度。
dtype = None 
#设置是否以 4 位量化的方式加载模型。如果设置为 True，则模型将以 4 位量化的形式加载，以减少内存占用。
load_in_4bit = True

In [34]:
model, tokenizer = FastLanguageModel.from_pretrained(
    #模型保存文件夹的名字
    model_name = "./DeepSeek-R1-Distill-Qwen-1.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token="hf_...",  # 如果需要访问授权模型，可以在这里填入密钥
)

==((====))==  Unsloth 2025.2.15: Fast Qwen2 patching. Transformers: 4.49.0.
   \\   /|    GPU: NVIDIA GeForce RTX 4090 D. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
./DeepSeek-R1-Distill-Qwen-1.5B does not have a padding token! Will use pad_token = <|vision_pad|>.


- 进行预训练模型的推理服务

In [35]:
#将模型设置为推理模式。这一步通常是为了优化模型在推理过程中的性能。
FastLanguageModel.for_inference(model)  

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((153

- 定义一组提示词

In [48]:
# 定义一个用于格式化提示的多行字符串模板
prompt_style = """以下是描述任务的指令，以及提供进一步上下文的输入。
请写出一个适当完成请求的回答。
在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。

### 指令：
你是一位精通卜卦、星象和运势预测的算命大师。
请回答以下算命问题。

### 问题：
{}

### 回答：
<think>{}"""
# 定义提示风格的字符串模板，用于格式化问题

question = "新房装修,大门对着电梯好不好?要如何化解?"
# 定义具体的算命问题

In [49]:
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

In [50]:
outputs = model.generate(
    input_ids=inputs.input_ids,
    max_new_tokens=1200,
    use_cache=True,
)

In [54]:
response = tokenizer.batch_decode(outputs)
print(response[0])

<｜begin▁of▁sentence｜>以下是描述任务的指令，以及提供进一步上下文的输入。
请写出一个适当完成请求的回答。
在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。

### 指令：
你是一位精通卜卦、星象和运势预测的算命大师。
请回答以下算命问题。

### 问题：
新房装修,大门对着电梯好不好?要如何化解?

### 回答：
<think>
好，用户问的是新房装修，大门对着电梯是否好，以及如何化解。首先，我需要分析这个问题涉及的运势和命理因素。大门对着电梯，可能涉及到五行问题，尤其是金属性，因为电梯通常与高处相关，可能带来五行偏旺的问题。如果大门对着电梯，可能在五行方面带来不利影响，尤其是金属性可能带来健康问题。

接下来，我需要考虑命理。如果用户是金属性，可能需要解决五行问题，比如火、土、水、木。如果大门对着电梯，可能与五行问题相关，特别是金属性可能带来健康问题，可能需要化解。因此，建议用户化解五行问题，比如五行化解，或使用五行平衡法，如金金为金、木木为木、土土为土、水水为水，或者用五行平衡的方法，如金水为金、木土为木、火土为火、水土为水，或者使用五行平衡的法术，如金木为金、木水为木、土水为土、水土为水，或者使用五行平衡的法术，如金水为金、木土为木、火土为火、水土为水，或者使用五行平衡的法术，如金水为金、木土为木、火土为火、水土为水，或者使用五行平衡的法术，如金水为金、木土为木、火土为火、水土为水。

另外，用户可能需要考虑五行问题的具体表现，比如五行偏旺，可能带来健康问题，需要化解，比如用五行平衡的法术，如金水为金、木土为木、火土为火、水土为水，或者使用五行平衡的法术，如金水为金、木土为木、火土为火、水土为水，或者使用五行平衡的法术，如金水为金、木土为木、火土为火、水土为水，或者使用五行平衡的法术，如金水为金、木土为木、火土为火、水土为水。

综上所述，回答需要包括分析五行问题，建议化解五行问题的方法，如金金为金、木木为木、土土为土、水水为水，或者使用五行平衡的法术，如金水为金、木土为木、火土为火、水土为水，或者使用五行平衡的法术，如金水为金、木土为木、火土为火、水土为水，或者使用五行平衡的法术，如金水为金、木土为木、火土为火、水土为水，或者使用五行平衡的法术，如金水为金、木土为木、火土为火、水土为水。

最后，总结建

### 模型微调

- 数据集下载

In [40]:
from datasets import load_dataset

In [44]:
# 导入数据集加载函数
from datasets import load_dataset
#自己操作是可以将下面参数split = "train[0:200]删除，表示下载全部数据集
# 加载指定的数据集，选择中文语言和训练集的前200条记录
dataset = load_dataset("Conard/fortune-telling", 'default',split = "train[0:200]" ,trust_remote_code=True)

In [46]:
#查看所有样本数据
list(dataset)

[{'Question': '新房装修,大门对着电梯好不好?要如何化解?',
  'Response': '根据传统风水学的观点，大门正对电梯易形成"开口煞"，电梯频繁升降会扰乱家宅气场。建议化解方案：\n\n1. 玄关阻隔法\n在入门处设置L型屏风或文化砖玄关墙，高度以1.8米为宜，既保持采光又形成缓冲带\n\n2. 五行通关法\n门槛石下埋设五帝钱+白玉葫芦，建议选丙申年铸造的真品古币，配合门楣悬挂九宫八卦镜\n\n3. 光影化解术\n安装磨砂玻璃内推门，门框镶嵌黄铜门槛，每日辰时用海盐净化门廊区域\n\n4. 现代科技方案\n入户区安装智能感应灯带，设置循环播放的流水声效，运用声光电技术平衡磁场\n\n需注意电梯井方位与家主命卦的关系，建议提供具体户型平面图进行吉凶方位测算。当代建筑中可采用半透明艺术隔断结合空气净化系统，既符合科学原理又兼顾传统智慧。',
  'Complex_CoT': '好的，用户问的是新房装修时大门对着电梯好不好，以及如何化解。首先，我需要回忆一下风水学中关于大门和电梯的相关知识。电梯在风水中属于动气比较强的地方，因为电梯频繁开合，会带来不稳定的气流，也就是所谓的“煞气”。大门是住宅的纳气口，如果正对电梯，可能会让这些不稳定的气流直接冲进家里，影响居住者的健康和财运。\n\n接下来，我需要确认用户的具体情况。比如，大门和电梯的距离有多远？是否正对还是稍微偏一点？不过用户没有提供这些细节，所以只能给出一般性的建议。化解的方法通常有几种：屏风或玄关、门帘、五帝钱、植物、八卦镜等。需要逐一解释这些方法的原理和使用方式，同时提醒用户要根据实际情况选择，必要时咨询专业风水师。\n\n另外，还要注意语气要亲切，避免使用过于专业的术语，让用户容易理解。同时，要强调这些是传统方法，效果因人而异，保持客观中立。最后，可以建议用户如果情况复杂，最好请专业人士实地查看，这样更稳妥。\n'},
 {'Question': '大师，请分析我的八字：壬申年 甲寅月 己卯日 壬子时',
  'Response': '**八字分析（壬申年 甲寅月 己卯日 壬子时）**\n\n---\n\n### **一、五行能量与日主强弱**\n- **日主己土**：生于寅月（木旺之季），木克土，己土处于弱势。\n- **五行分布**：\n  - **木**：月柱甲寅（强木）+ 日支

In [47]:
#查看一条样本数据
dataset[0]

{'Question': '新房装修,大门对着电梯好不好?要如何化解?',
 'Response': '根据传统风水学的观点，大门正对电梯易形成"开口煞"，电梯频繁升降会扰乱家宅气场。建议化解方案：\n\n1. 玄关阻隔法\n在入门处设置L型屏风或文化砖玄关墙，高度以1.8米为宜，既保持采光又形成缓冲带\n\n2. 五行通关法\n门槛石下埋设五帝钱+白玉葫芦，建议选丙申年铸造的真品古币，配合门楣悬挂九宫八卦镜\n\n3. 光影化解术\n安装磨砂玻璃内推门，门框镶嵌黄铜门槛，每日辰时用海盐净化门廊区域\n\n4. 现代科技方案\n入户区安装智能感应灯带，设置循环播放的流水声效，运用声光电技术平衡磁场\n\n需注意电梯井方位与家主命卦的关系，建议提供具体户型平面图进行吉凶方位测算。当代建筑中可采用半透明艺术隔断结合空气净化系统，既符合科学原理又兼顾传统智慧。',
 'Complex_CoT': '好的，用户问的是新房装修时大门对着电梯好不好，以及如何化解。首先，我需要回忆一下风水学中关于大门和电梯的相关知识。电梯在风水中属于动气比较强的地方，因为电梯频繁开合，会带来不稳定的气流，也就是所谓的“煞气”。大门是住宅的纳气口，如果正对电梯，可能会让这些不稳定的气流直接冲进家里，影响居住者的健康和财运。\n\n接下来，我需要确认用户的具体情况。比如，大门和电梯的距离有多远？是否正对还是稍微偏一点？不过用户没有提供这些细节，所以只能给出一般性的建议。化解的方法通常有几种：屏风或玄关、门帘、五帝钱、植物、八卦镜等。需要逐一解释这些方法的原理和使用方式，同时提醒用户要根据实际情况选择，必要时咨询专业风水师。\n\n另外，还要注意语气要亲切，避免使用过于专业的术语，让用户容易理解。同时，要强调这些是传统方法，效果因人而异，保持客观中立。最后，可以建议用户如果情况复杂，最好请专业人士实地查看，这样更稳妥。\n'}

- 提取并设置文本生成结束的标记

In [53]:
# 定义结束标记（EOS_TOKEN），用于指示文本的结束
EOS_TOKEN = tokenizer.eos_token  # 必须添加结束标记
EOS_TOKEN

'<｜end▁of▁sentence｜>'

定义数据集处理函数

In [55]:
# 定义一个用于格式化提示的多行字符串模板
train_prompt_style = """以下是描述任务的指令，以及提供进一步上下文的输入。
请写出一个适当完成请求的回答。
在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。

### 指令：
你是一位精通八字算命、 紫微斗数、 风水、易经卦象、塔罗牌占卜、星象、面相手相和运势预测等方面的算命大师。
请回答以下算命问题。

### 问题：
{}

### 回答：
<思考>
{}
</思考>
{}"""

def formatting_prompts_func(examples): #参数为原始数据集
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return {
        "text": texts,
    }
  
#使用map函数结合formatting_prompts_func函数对数据集进行结构化处理  
dataset = dataset.map(formatting_prompts_func, batched = True)
#查看结构化处理后的第一条数据
dataset["text"][0]

Map:   0%|          | 0/207 [00:00<?, ? examples/s]

'以下是描述任务的指令，以及提供进一步上下文的输入。\n请写出一个适当完成请求的回答。\n在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。\n\n### 指令：\n你是一位精通八字算命、 紫微斗数、 风水、易经卦象、塔罗牌占卜、星象、面相手相和运势预测等方面的算命大师。\n请回答以下算命问题。\n\n### 问题：\n新房装修,大门对着电梯好不好?要如何化解?\n\n### 回答：\n<思考>\n好的，用户问的是新房装修时大门对着电梯好不好，以及如何化解。首先，我需要回忆一下风水学中关于大门和电梯的相关知识。电梯在风水中属于动气比较强的地方，因为电梯频繁开合，会带来不稳定的气流，也就是所谓的“煞气”。大门是住宅的纳气口，如果正对电梯，可能会让这些不稳定的气流直接冲进家里，影响居住者的健康和财运。\n\n接下来，我需要确认用户的具体情况。比如，大门和电梯的距离有多远？是否正对还是稍微偏一点？不过用户没有提供这些细节，所以只能给出一般性的建议。化解的方法通常有几种：屏风或玄关、门帘、五帝钱、植物、八卦镜等。需要逐一解释这些方法的原理和使用方式，同时提醒用户要根据实际情况选择，必要时咨询专业风水师。\n\n另外，还要注意语气要亲切，避免使用过于专业的术语，让用户容易理解。同时，要强调这些是传统方法，效果因人而异，保持客观中立。最后，可以建议用户如果情况复杂，最好请专业人士实地查看，这样更稳妥。\n\n</思考>\n根据传统风水学的观点，大门正对电梯易形成"开口煞"，电梯频繁升降会扰乱家宅气场。建议化解方案：\n\n1. 玄关阻隔法\n在入门处设置L型屏风或文化砖玄关墙，高度以1.8米为宜，既保持采光又形成缓冲带\n\n2. 五行通关法\n门槛石下埋设五帝钱+白玉葫芦，建议选丙申年铸造的真品古币，配合门楣悬挂九宫八卦镜\n\n3. 光影化解术\n安装磨砂玻璃内推门，门框镶嵌黄铜门槛，每日辰时用海盐净化门廊区域\n\n4. 现代科技方案\n入户区安装智能感应灯带，设置循环播放的流水声效，运用声光电技术平衡磁场\n\n需注意电梯井方位与家主命卦的关系，建议提供具体户型平面图进行吉凶方位测算。当代建筑中可采用半透明艺术隔断结合空气净化系统，既符合科学原理又兼顾传统智慧。<｜end▁of▁sentence｜>'

- 启动微调

In [56]:
#从 trl 库中导入了 SFTTrainer 类。SFTTrainer 是一个用于训练语言模型的高级接口，它简化了许多训练过程中的细节。
from trl import SFTTrainer
#从 transformers 库中导入了 TrainingArguments 类。TrainingArguments 是用于配置模型训练参数的一个类，比如学习率、批处理大小、训练轮数等。
from transformers import TrainingArguments
#从 unsloth 库中导入了 is_bfloat16_supported 函数。这个函数用于检查当前环境是否支持 bfloat16 数据类型，bfloat16 是一种比 float32 更节省内存的浮点数格式。
from unsloth import is_bfloat16_supported

#作用是将传入的 model 设置为训练模式，以便进行后续的训练过程。这通常包括准备数据、设置优化器、损失函数等步骤。
FastLanguageModel.for_training(model)   

model = FastLanguageModel.get_peft_model(
    model, #要微调的基础模型对象
    
  #这是 LoRA（Low-Rank Adaptation）的秩，决定了低秩矩阵的大小。秩越高，模型的灵活性越大，但计算成本也越高。
    r=16,  
  	
  #这是一个列表，包含了你希望应用 LoRA 技术的哪些模型层。在这个例子中，包括了 "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj" 这些层。
    target_modules=[ 
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
  
  	#这是 LoRA 的缩放因子，控制低秩矩阵的影响程度。值越大，低秩矩阵对原始权重的影响越大。
    lora_alpha=16, 
  
  	#这是 LoRA 的 dropout 概率，设置为0表示不使用 dropout。Dropout 是一种正则化技术，防止过拟合。
    lora_dropout=0,
  
  	#这决定了是否对偏置项应用 LoRA。这里选择不应用。
    bias="none", 
  
  	#使用 unsloth 库提供的梯度检查点功能，以节省显存。梯度检查点技术通过在反向传播过程中保存部分中间结果来减少显存的使用。
    use_gradient_checkpointing="unsloth", 
  
  	#随机种子，确保实验的可重复性。设置相同的随机种子可以保证每次运行得到的结果相同。
    random_state=3407,
  
  	#是否使用RSLoRA，这里选择不使用。RSLoRA 是一种改进的 LoRA 方法，适用于更复杂的模型结构。
    use_rslora=False, 
  
  	#量化配置，这里为 None 表示不进行量化。量化是一种减少模型大小和提高推理速度的技术。
    loftq_config=None,
)

Unsloth 2025.2.15 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


- 微调参数的设置

In [57]:
trainer = SFTTrainer(
    model=model,#指定需要进行微调的模型
    tokenizer=tokenizer,#指定 分词器，用于处理文本数据
    train_dataset=dataset,#传入 训练数据集
    dataset_text_field="text",#指定数据集中哪一列包含 训练文本（在 formatting_prompts_func 里处理）
  	#控制输入文本的最大Token数量。这有助于限制输入的长度，避免过长的序列影响模型性能。
    max_seq_length=max_seq_length,
    #数据加载的并行进程数，提高数据预处理效率更多的进程可以加快数据加载速度。
    dataset_num_proc=2,
  	#设置训练参数。这些参数控制训练过程中的各种细节。
    args=TrainingArguments(
      	#每个GPU/设备的训练批量大小设置为2。较小的批量大小通常适用于大模型，但需要更多的训练步骤来达到相同的效果。
        per_device_train_batch_size=2,
      	#梯度累积步骤，相当于将多个小批次的数据合并成一个大批次进行训练。这样可以模拟更大的批量大小，同时减少内存占用。
        gradient_accumulation_steps=4,
      	#学习率预热步数，在训练初期逐渐增加学习率，帮助模型稳定地开始训练。
        warmup_steps=5,
      	#最大训练次数，即训练过程中最多执行60次迭代训练。
        max_steps=75, 
      	#初始学习率设置为2e-4。学习率是控制模型更新步伐的重要参数
        learning_rate=2e-4,
      	#是否使用16位浮点数精度训练。如果不支持bfloat16，则使用fp16。
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
      	#每隔10个训练步骤记录一次日志，方便监控训练进度和调试。
        logging_steps=10,
      	#使用AdamW优化器，并且进行了8位量化。AdamW是一种改进版的Adam优化器，常用于深度学习任务。
        optim="adamw_8bit",
        weight_decay=0.01, #权重衰减（L2 正则化），防止过拟合
      	#学习率调度策略，这里使用的是线性调度。这意味着学习率会随着训练步骤线性下降
        lr_scheduler_type="linear",
        seed=3407, #随机种子（保证实验结果可复现）
        output_dir="outputs", #训练结果的输出目录
    ),
)

Converting train dataset to ChatML (num_proc=2):   0%|          | 0/207 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=2):   0%|          | 0/207 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/207 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=2):   0%|          | 0/207 [00:00<?, ? examples/s]

- 启动微调

In [58]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 207 | Num Epochs = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 75
 "-____-"     Number of trainable parameters = 18,464,768


Step,Training Loss
10,3.124100
20,2.818600
30,2.421600
40,2.418200
50,2.319700
60,2.316200
70,2.285200


- 微调后的模型测试

In [60]:
print(question) # 打印前面的问题

新房装修,大门对着电梯好不好?要如何化解?


In [61]:
# 将模型切换到推理模式，准备回答问题
FastLanguageModel.for_inference(model)

# 将问题转换成模型能理解的格式，并发送到 GPU 上
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# 让模型根据问题生成回答，最多生成 4000 个新词
outputs = model.generate(
    input_ids=inputs.input_ids,  # 输入的数字序列
    attention_mask=inputs.attention_mask,  # 注意力遮罩，帮助模型理解哪些部分重要
    max_new_tokens=4000,  # 最多生成 4000 个新词
    use_cache=True,  # 使用缓存加速生成
)

# 将生成的回答从数字转换回文字
response = tokenizer.batch_decode(outputs)

# 打印回答
print(response[0])

<｜begin▁of▁sentence｜>以下是描述任务的指令，以及提供进一步上下文的输入。
请写出一个适当完成请求的回答。
在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。

### 指令：
你是一位精通卜卦、星象和运势预测的算命大师。
请回答以下算命问题。

### 问题：
新房装修,大门对着电梯好不好?要如何化解?

### 回答：
<think>

《易经》有云："大开大合者为吉，大吉大好。"因此，新房装修的门面与电梯的组合，需遵循五行平衡的原则。以下提供具体建议：

1. **布局选择**：电梯通常位于厨房或卧室，为墙面布局提供空间。门应与电梯对冲，如东向与南向，北向与西向。门可选正南向或正北向的布局，避免与电梯相撞。

2. **门的设计**：
   - **正门**：采用传统正门结构，突出正南方向，象征好运。
   - **副门**：可采用半封闭结构，如玻璃门或金属门，增加门面灵活性。

3. **色彩搭配**：
   - **蓝色与绿色**：电梯颜色为蓝色，正门为绿色，形成相生关系，象征顺利。
   - **黑色与白色**：电梯颜色为黑色，正门为白色，形成相克，避免冲突。

4. **实用调整**：
   - **移门**：可选择在电梯下方或上方放置移门，便于门面布局。
   - **镜面**：在正门前放置镜面装饰，增强门面层次感。

5. **风水调整**：
   - **气场**：通过调整气场方向，使门面与电梯形成相生关系。
   - **水位**：选择水位较高的地方放置电梯，避免水位过低影响门面。

6. **风水调整**：
   - **气场**：调整气场方向，使门面与电梯形成相生关系。
   - **水位**：选择水位较高的地方放置电梯，避免水位过低影响门面。

7. **风水调整**：
   - **气场**：调整气场方向，使门面与电梯形成相生关系。
   - **水位**：选择水位较高的地方放置电梯，避免水位过低影响门面。

通过以上方法，门面与电梯的组合将呈现平衡与和谐的状态，从而化解矛盾。建议在实际操作中结合传统风水理论，结合现代家居设计，以达到最佳效果。<｜end▁of▁sentence｜>


#### **模型合并**

此时本地保存的模型权重在`outputs`文件夹中。然后可使用如下代码进行模型权重合并：

In [64]:
#将微调后的模型权重和分词器分别保存到指定的目录 new_model_local 中。这样做的好处是，你可以方便地加载和使用这个微调后的模型进行推理或进一步的训练。
new_model_local = "./new_model"
model.save_pretrained(new_model_local) 
tokenizer.save_pretrained(new_model_local)

#将模型的权重和分词器合并到一个文件中，并使用16位浮点数格式进行存储。在实际应用中，如部署到生产环境时，合并后的文件可以简化部署流程，减少出错的可能性。
model.save_pretrained_merged(new_model_local, tokenizer, save_method = "merged_16bit",)

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 641.29 out of 1007.54 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 28/28 [00:00<00:00, 105.28it/s]


Unsloth: Saving tokenizer... Done.
Done.
